# 🌊 YOLO-UDD v2.0 - Google Drive Training

**Complete training notebook for Google Colab with your dataset in Google Drive**

Repository: https://github.com/kshitijkhede/YOLO-UDD-v2.0

---

## 📋 Prerequisites
Your Google Drive should have:
```
My Drive/
  └── TrashCan_dataset/
      ├── original_data/
      │   ├── images/
      │   └── annotations/
      └── scripts/
          └── trash_can_coco.py
```

## Step 1: Clone Repository & Setup

In [ ]:
!git clone https://github.com/kshitijkhede/YOLO-UDD-v2.0.git
%cd YOLO-UDD-v2.0
print("✅ Repository cloned!")

## Step 2: Install Dependencies

In [ ]:
!pip install -q torch torchvision albumentations opencv-python-headless tqdm pyyaml tensorboard pycocotools
print("✅ All dependencies installed!")

## Step 3: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted!")

## Step 4: Convert Original Data to COCO Format

Your dataset has individual JSON files per image. We need to convert them to COCO format first.

In [ ]:
import os
import json
import glob
from pathlib import Path

# Set your Google Drive paths
DATASET_ROOT = '/content/drive/MyDrive/TrashCan_dataset'
ORIGINAL_DATA = os.path.join(DATASET_ROOT, 'original_data')
IMAGES_DIR = os.path.join(ORIGINAL_DATA, 'images')
ANNOTATIONS_DIR = os.path.join(ORIGINAL_DATA, 'annotations')

print("Checking dataset structure...")
print(f"Dataset root: {DATASET_ROOT}")
print(f"Images: {IMAGES_DIR}")
print(f"Annotations: {ANNOTATIONS_DIR}")

if not os.path.exists(IMAGES_DIR):
    print("❌ ERROR: Images folder not found!")
    print(f"Expected: {IMAGES_DIR}")
else:
    image_files = [f for f in os.listdir(IMAGES_DIR) if f.endswith('.jpg')]
    print(f"✅ Found {len(image_files)} images")

if not os.path.exists(ANNOTATIONS_DIR):
    print("❌ ERROR: Annotations folder not found!")
    print(f"Expected: {ANNOTATIONS_DIR}")
else:
    ann_files = [f for f in os.listdir(ANNOTATIONS_DIR) if f.endswith('.json')]
    print(f"✅ Found {len(ann_files)} annotation files")

## Step 5: Create COCO Format Annotations

In [ ]:
import random
from PIL import Image

def convert_to_coco_format(images_dir, annotations_dir, output_file, split='train', train_ratio=0.8):
    """
    Convert TrashCAN original format to COCO format
    """
    coco_data = {
        "images": [],
        "annotations": [],
        "categories": [
            {"id": 1, "name": "trash"},
            {"id": 2, "name": "animal"},
            {"id": 3, "name": "rov"}
        ]
    }
    
    # Get all image files
    image_files = sorted([f for f in os.listdir(images_dir) if f.endswith('.jpg')])
    
    # Split into train/val
    random.seed(42)
    random.shuffle(image_files)
    split_idx = int(len(image_files) * train_ratio)
    
    if split == 'train':
        selected_files = image_files[:split_idx]
    else:  # val
        selected_files = image_files[split_idx:]
    
    annotation_id = 1
    
    for image_id, img_filename in enumerate(selected_files, start=1):
        # Get image dimensions
        img_path = os.path.join(images_dir, img_filename)
        try:
            with Image.open(img_path) as img:
                width, height = img.size
        except:
            print(f"Warning: Could not read {img_filename}")
            continue
        
        # Add image info
        coco_data["images"].append({
            "id": image_id,
            "file_name": img_filename,
            "width": width,
            "height": height
        })
        
        # Load annotations
        ann_filename = img_filename + '.json'
        ann_path = os.path.join(annotations_dir, ann_filename)
        
        if os.path.exists(ann_path):
            try:
                with open(ann_path, 'r') as f:
                    ann_data = json.load(f)
                
                # Extract bounding boxes
                for obj in ann_data.get('objects', []):
                    # Get category
                    category_name = obj.get('classTitle', 'trash').lower()
                    if 'trash' in category_name:
                        category_id = 1
                    elif 'animal' in category_name:
                        category_id = 2
                    elif 'rov' in category_name:
                        category_id = 3
                    else:
                        category_id = 1  # Default to trash
                    
                    # Get bounding box (assuming 'points' field with polygon/bbox)
                    if 'points' in obj:
                        points = obj['points']['exterior']
                        xs = [p[0] for p in points]
                        ys = [p[1] for p in points]
                        
                        x_min, x_max = min(xs), max(xs)
                        y_min, y_max = min(ys), max(ys)
                        
                        bbox_width = x_max - x_min
                        bbox_height = y_max - y_min
                        
                        if bbox_width > 0 and bbox_height > 0:
                            coco_data["annotations"].append({
                                "id": annotation_id,
                                "image_id": image_id,
                                "category_id": category_id,
                                "bbox": [x_min, y_min, bbox_width, bbox_height],
                                "area": bbox_width * bbox_height,
                                "iscrowd": 0
                            })
                            annotation_id += 1
            except Exception as e:
                print(f"Warning: Error processing {ann_filename}: {e}")
    
    # Save COCO format
    with open(output_file, 'w') as f:
        json.dump(coco_data, f)
    
    print(f"✅ Created {output_file}")
    print(f"   Images: {len(coco_data['images'])}")
    print(f"   Annotations: {len(coco_data['annotations'])}")
    
    return coco_data

# Create COCO format files
print("Converting to COCO format...")
print()

train_output = os.path.join(ORIGINAL_DATA, 'instances_train_trashcan.json')
val_output = os.path.join(ORIGINAL_DATA, 'instances_val_trashcan.json')

train_data = convert_to_coco_format(IMAGES_DIR, ANNOTATIONS_DIR, train_output, split='train', train_ratio=0.8)
print()
val_data = convert_to_coco_format(IMAGES_DIR, ANNOTATIONS_DIR, val_output, split='val', train_ratio=0.8)

print("\n✅ COCO format conversion complete!")

## Step 6: Check GPU & Verify Dataset

In [ ]:
import torch

print("=" * 60)
print("GPU Check:")
print("=" * 60)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("⚠️  WARNING: No GPU detected!")
    print("   Go to: Runtime → Change runtime type → T4 GPU")

print("\n" + "=" * 60)
print("Dataset Verification:")
print("=" * 60)

train_json = os.path.join(ORIGINAL_DATA, 'instances_train_trashcan.json')
val_json = os.path.join(ORIGINAL_DATA, 'instances_val_trashcan.json')

if os.path.exists(train_json):
    with open(train_json, 'r') as f:
        train_data = json.load(f)
    print(f"✅ Training set: {len(train_data['images'])} images, {len(train_data['annotations'])} annotations")
else:
    print("❌ Training JSON not found!")

if os.path.exists(val_json):
    with open(val_json, 'r') as f:
        val_data = json.load(f)
    print(f"✅ Validation set: {len(val_data['images'])} images, {len(val_data['annotations'])} annotations")
else:
    print("❌ Validation JSON not found!")

print("\n✅ Ready to train!")

## Step 7: Configure Training Parameters

In [ ]:
# Training configuration
BATCH_SIZE = 8
EPOCHS = 50
LEARNING_RATE = 0.01
IMAGE_SIZE = 640
SAVE_DIR = 'runs/train'

print("Training Configuration:")
print(f"  Dataset: {ORIGINAL_DATA}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Image Size: {IMAGE_SIZE}")
print(f"  Save Directory: {SAVE_DIR}")

## Step 8: Start Training 🚀

In [ ]:
# Run training
!python scripts/train.py \
  --data-dir "{ORIGINAL_DATA}" \
  --batch-size {BATCH_SIZE} \
  --epochs {EPOCHS} \
  --lr {LEARNING_RATE} \
  --img-size {IMAGE_SIZE} \
  --save-dir {SAVE_DIR}

## Step 9: View Training Results

In [ ]:
import glob
from IPython.display import Image as IPImage, display

print("=" * 60)
print("Training Results:")
print("=" * 60)

# Check for saved checkpoints
checkpoint_dir = os.path.join(SAVE_DIR, 'checkpoints')
if os.path.exists(checkpoint_dir):
    checkpoints = glob.glob(os.path.join(checkpoint_dir, '*.pt'))
    if checkpoints:
        print(f"\n✅ Found {len(checkpoints)} checkpoint(s):")
        for ckpt in checkpoints:
            size = os.path.getsize(ckpt) / (1024*1024)
            print(f"   📦 {os.path.basename(ckpt)} ({size:.1f} MB)")
    else:
        print("\n❌ No checkpoints found")
else:
    print("\n❌ Checkpoint directory not found")

# Display training curves if available
plots_dir = os.path.join(SAVE_DIR, 'plots')
if os.path.exists(plots_dir):
    print("\n📊 Training Plots:")
    for plot_file in glob.glob(os.path.join(plots_dir, '*.png')):
        print(f"\n{os.path.basename(plot_file)}:")
        display(IPImage(filename=plot_file, width=800))

## Step 10: Download Best Model

In [ ]:
from google.colab import files

best_model_path = os.path.join(SAVE_DIR, 'checkpoints', 'best_model.pt')

if os.path.exists(best_model_path):
    print(f"Downloading best model from: {best_model_path}")
    files.download(best_model_path)
    print("✅ Download complete!")
else:
    print("❌ Best model not found!")
    print(f"Expected location: {best_model_path}")

## Step 11: Save Results to Google Drive

In [ ]:
import shutil

# Copy results to Google Drive
drive_results_dir = os.path.join(DATASET_ROOT, 'training_results')
os.makedirs(drive_results_dir, exist_ok=True)

if os.path.exists(SAVE_DIR):
    print(f"Copying results to: {drive_results_dir}")
    
    # Copy checkpoints
    if os.path.exists(checkpoint_dir):
        shutil.copytree(checkpoint_dir, os.path.join(drive_results_dir, 'checkpoints'), dirs_exist_ok=True)
        print("✅ Checkpoints saved to Google Drive")
    
    # Copy plots
    if os.path.exists(plots_dir):
        shutil.copytree(plots_dir, os.path.join(drive_results_dir, 'plots'), dirs_exist_ok=True)
        print("✅ Plots saved to Google Drive")
    
    # Copy logs
    log_files = glob.glob(os.path.join(SAVE_DIR, '*.log')) + glob.glob(os.path.join(SAVE_DIR, '*.txt'))
    for log_file in log_files:
        shutil.copy2(log_file, drive_results_dir)
    print("✅ Logs saved to Google Drive")
    
    print(f"\n✅ All results saved to: {drive_results_dir}")
else:
    print("❌ No results directory found")